# Assignment 5 — RAG Agent

This notebook implements a RAG Agent that answers multi-step questions using a different data source from previous assignments.

Data source used:
Official Python documentation web pages

## Overview

This notebook follows a RAG pipeline:

1. Load documents from web pages
2. Split them into chunks
3. Generate embeddings
4. Store them in a vector store
5. Create a retrieval tool
6. Build a RAG Agent that uses the retrieval tool to answer multi-step questions

In [2]:
%pip install -q langchain langchain-core "langchain[openai]" langchain-text-splitters langchain-community bs4 langchain-huggingface sentence-transformers

## Load the OpenRouter API Key

In [3]:
import os

try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()
    print("OPENROUTER_API_KEY loaded successfully")
except Exception as e:
    print("Error loading API key:", e)

OPENROUTER_API_KEY loaded successfully


## Create the Chat Model

In [4]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

## Create the Embedding Model and Vector Store

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

vector_store = InMemoryVectorStore(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Load Documents from a New Data Source

We use web pages from the official Python documentation.

In [6]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://docs.python.org/3/tutorial/classes.html",
    "https://docs.python.org/3/tutorial/errors.html",
    "https://docs.python.org/3/tutorial/datastructures.html",
]

loader = WebBaseLoader(
    web_paths=urls,
    bs_kwargs={"parse_only": bs4.SoupStrainer(["div", "main", "section", "article"])}
)

docs = loader.load()

print("Loaded documents:", len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:500])

Loaded documents: 3
{'source': 'https://docs.python.org/3/tutorial/classes.html'}
























    Theme
    
Auto
Light
Dark



Table of Contents

9. Classes
9.1. A Word About Names and Objects
9.2. Python Scopes and Namespaces
9.2.1. Scopes and Namespaces Example


9.3. A First Look at Classes
9.3.1. Class Definition Syntax
9.3.2. Class Objects
9.3.3. Instance Objects
9.3.4. Method Objects
9.3.5. Class and Instance Variables


9.4. Random Remarks
9.5. Inheritance
9.5.1. Multiple Inheritance


9.6. Private Variables
9.7. Odds and Ends
9.8. Iterators
9.9. Generator


## Split the Documents into Chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

print("Total chunks:", len(all_splits))
print(all_splits[0].metadata)
print(all_splits[0].page_content[:500])

Total chunks: 120
{'source': 'https://docs.python.org/3/tutorial/classes.html', 'start_index': 28}
Theme
    
Auto
Light
Dark



Table of Contents

9. Classes
9.1. A Word About Names and Objects
9.2. Python Scopes and Namespaces
9.2.1. Scopes and Namespaces Example


9.3. A First Look at Classes
9.3.1. Class Definition Syntax
9.3.2. Class Objects
9.3.3. Instance Objects
9.3.4. Method Objects
9.3.5. Class and Instance Variables


9.4. Random Remarks
9.5. Inheritance
9.5.1. Multiple Inheritance


9.6. Private Variables
9.7. Odds and Ends
9.8. Iterators
9.9. Generators
9.10. Generator Expression


## Store the Chunks in the Vector Store

In [8]:
document_ids = vector_store.add_documents(documents=all_splits)

print("Stored chunks:", len(document_ids))
print(document_ids[:3])

Stored chunks: 120
['8732a28b-0b67-4016-a6b3-5be9aa648745', '134f9ba3-8d6e-4305-9871-1eff66400436', 'c55f39d7-7aaa-43d6-bb70-ff7fc8682282']


## Create a Retrieval Tool

This tool retrieves relevant document chunks from the vector store to help answer the question.

In [9]:
from langchain.tools import tool

@tool
def retrieve_context(query: str) -> str:
    """Retrieve relevant context from the indexed documents."""
    retrieved_docs = vector_store.similarity_search(query, k=3)
    serialized = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized

## Create the RAG Agent

In [10]:
from langchain.agents import create_agent

RAG_AGENT_PROMPT = """
You are a helpful RAG agent.

Your job is to answer user questions using the retrieve_context tool.

Instructions:
1. Use the retrieve_context tool to search the indexed documents.
2. Use the retrieved information to answer the question.
3. For multi-step questions, combine the relevant pieces of information into one clear answer.
4. Keep your final answer clear and concise.
5. If the retrieved context is not enough, say so clearly.
"""

rag_agent = create_agent(
    model=model,
    tools=[retrieve_context],
    system_prompt=RAG_AGENT_PROMPT
)

## Test 1 — Simple Question

In [13]:
response1 = rag_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is a Python class?"
            }
        ]
    }
)

print(response1["messages"][-1].content)

A Python class is a blueprint that defines a new type of object. It bundles together data (attributes) and functionality (methods) so that each instance of the class can maintain its own state and behavior. Creating a class lets you instantiate objects that share the same structure and capabilities, enabling organized and reusable code.


## Test 2 — Multi-Step Question

In [14]:
response2 = rag_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the difference between Python classes and lists, and how are exceptions handled?"
            }
        ]
    }
)

print(response2["messages"][-1].content)

**Python classes vs. lists**

- **Classes** are *user‑defined types* that bundle data (attributes) and behavior (methods) together.  
  - You create a class with `class MyClass: …`, then instantiate it to get objects that can hold their own state and implement custom methods.  
  - Classes support inheritance, encapsulation, and polymorphism – the core ideas of object‑oriented programming.  

- **Lists** are a *built‑in mutable sequence type* used to store an ordered collection of items.  
  - A list is itself an instance of the built‑in `list` class, but it does **not** let you define new behavior or state beyond the methods that the list class already provides.  
  - Lists are primarily a data‑container; they are not a way to define a new object type with its own identity and methods.  

In short, a class is a blueprint for creating objects with custom data and functionality, whereas a list is a ready‑made container for holding multiple values.

---

**How exceptions are handled**

-

## Why This is a RAG Agent

This notebook does not answer from the model alone.
Instead, the agent first retrieves relevant context from the indexed documents, then uses that retrieved context to generate the answer.

This is Retrieval-Augmented Generation (RAG), and the use of a retrieval tool inside an agent makes it a RAG Agent.

## Conclusion

This notebook implemented a RAG Agent using a new data source: official Python documentation web pages.

The system:
- loaded web documents
- split them into chunks
- embedded and stored them
- retrieved relevant context
- used an agent to answer multi-step questions